# Image binarization with a time-fractional diffusion-reaction equation

We treat the image as the initial condition of

$$D_t^\alpha u = \Delta u + f(u), \qquad u(x,y,0) = \text{image}$$

- **Diffusion** ($\Delta u$) smooths noise.
- **Source term** $f(u)$ pushes pixels toward 0 (black) or 1 (white), which is what binarizes.
- **Fractional order** $\alpha \in (0,1]$ adds memory. $\alpha = 1$ is ordinary diffusion.

Method: finite differences in space (5-point stencil, Neumann boundaries) and the **L1 scheme** for the Caputo derivative in time.

**Step 1: imports.** NumPy does the array math, `gamma` is needed for the L1 scheme constant, Matplotlib plots, and PIL loads images.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import gamma
from PIL import Image

**Step 2: normalization and the Laplacian.**
`normalize` rescales pixel values to [0, 1] so they live on the same scale as the stable states of $f$.
`laplacian_neumann` computes the 5-point stencil: (up + down + left + right − 4·center) / h².
Padding the border with `mode="edge"` copies edge pixels outward, which gives zero flux (Neumann) at the boundary, so nothing leaks out of the image.

In [ ]:
def normalize(img):
    img = img.astype(float)
    return (img - img.min()) / (img.max() - img.min() + 1e-12)

def laplacian_neumann(u, h=1.0):
    p = np.pad(u, 1, mode="edge")
    return (p[:-2, 1:-1] + p[2:, 1:-1] + p[1:-1, :-2] + p[1:-1, 2:] - 4*u) / h**2

**Step 3: the source term $f(u)$.**
This is a **placeholder** bistable cubic, $f(u)=u(1-u)(u-\theta)$. It has stable points at 0 and 1 and an unstable point at $\theta$. Pixels above $\theta$ get pushed to 1 and pixels below get pushed to 0.
**Replace this with the paper's $f(u)$** once you find it on an earlier page.

In [ ]:
def f(u, theta=0.5):
    return u * (1 - u) * (u - theta)

**Step 4: the L1 time-stepping scheme.**
The Caputo derivative at step $n+1$ is approximated by a weighted sum of *all* past increments:

$$D_t^\alpha u^{n+1} \approx c \sum_{k=0}^{n} b_k\,(u^{n+1-k}-u^{n-k}), \quad b_k=(k+1)^{1-\alpha}-k^{1-\alpha}, \quad c=\frac{\Delta t^{-\alpha}}{\Gamma(2-\alpha)}$$

Setting this equal to $\Delta u^n + f(u^n)$ (explicit) and solving for $u^{n+1}$ gives the update in the loop. The `mem` variable is the **memory term**. It is what makes the method fractional.

**Stability:** the effective step is $\Delta t^\alpha\Gamma(2-\alpha)$, which must be small (about < 0.1 for h = 1). So `dt_from_lambda` chooses $\Delta t$ for each $\alpha$ to keep that effective step fixed at `lam`. If you see NaNs, lower `lam`.

`window` optionally truncates the memory to the last N steps, to save time and RAM on big images.

In [ ]:
def dt_from_lambda(alpha, lam):
    return (lam / gamma(2 - alpha)) ** (1 / alpha)

def solve(u0, alpha, dt, n_steps, h=1.0, theta=0.5, window=None):
    c = dt ** (-alpha) / gamma(2 - alpha)
    k = np.arange(n_steps + 1)
    b = (k + 1) ** (1 - alpha) - k ** (1 - alpha)
    u = u0.copy()
    diffs = []                 # stores u^{j+1} - u^j
    traj = [u.copy()]
    for n in range(n_steps):
        rhs = laplacian_neumann(u, h) + f(u, theta)
        hist = diffs if window is None else diffs[-window:]
        mem = np.zeros_like(u)
        for kk in range(1, len(hist) + 1):
            mem += b[kk] * hist[-kk]
        u_new = u - mem + rhs / c
        diffs.append(u_new - u)
        u = u_new
        traj.append(u.copy())
    return traj

**Step 5: the error metric.**
We binarize $u$ at 0.5 and compare it with the binary ground truth. The result is the **percentage of wrong pixels**, a mean absolute (L1) error, which we track over time like Fig. 8 in the paper.

In [ ]:
def l1_error(u, gt, thresh=0.5):
    return 100 * np.mean(np.abs((u > thresh).astype(float) - gt))

**Step 6: load the data.**
Set `USE_UPLOAD = True` to upload your own noisy image and a ground-truth image (same content, clean and black/white). Otherwise a synthetic noisy letter is used so you can test everything first.
Images are shrunk to at most `MAX_SIZE` pixels on the long side because the memory term stores every step (cost grows with image size and step count).

In [ ]:
USE_UPLOAD = False
MAX_SIZE = 96

def load_gray(path, size=None):
    im = Image.open(path).convert("L")
    if size is not None:
        im = im.resize(size)
    return np.array(im)

if USE_UPLOAD:
    from google.colab import files
    print("Upload the NOISY image, then the GROUND TRUTH image:")
    up = files.upload()
    names = list(up.keys())
    im = Image.open(names[0]).convert("L")
    s = MAX_SIZE / max(im.size)
    size = (int(im.size[0]*s), int(im.size[1]*s))
    noisy = load_gray(names[0], size)
    gt = (normalize(load_gray(names[1], size)) > 0.5).astype(float)
    # make sure text is 1 and background 0 in BOTH images
    if gt.mean() > 0.5: gt = 1 - gt
    u0 = normalize(noisy)
    if u0.mean() > 0.5: u0 = 1 - u0   # text -> high values
else:
    rng = np.random.default_rng(0)
    gt = np.zeros((48, 48))
    gt[10:38, 8:12] = 1; gt[10:14, 8:30] = 1; gt[22:26, 8:26] = 1
    noisy = 0.7*gt + 0.15 + 0.15*rng.standard_normal(gt.shape)
    u0 = normalize(noisy)

fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].imshow(u0, cmap="gray"); ax[0].set_title("u0 (input)")
ax[1].imshow(gt, cmap="gray"); ax[1].set_title("ground truth")
for a in ax: a.axis("off")
plt.show()

**Step 7: sweep $\alpha$.**
For each $\alpha$ we run the solver and record the L1 error at every step. We print the best error and the step where it occurs (the ideal **stopping time**), plus the final error, to see how much it drifts if we run too long.

In [ ]:
alphas = [1, 0.8, 0.75, 0.6, 0.5, 0.25]
LAM, N_STEPS = 0.05, 200

results = {}
for a in alphas:
    dt = dt_from_lambda(a, LAM)
    traj = solve(u0, a, dt, N_STEPS)
    errs = np.array([l1_error(u, gt) for u in traj])
    results[a] = (traj, errs)
    print(f"alpha={a:<5} best={errs.min():6.2f}% at step {errs.argmin():3d}   final={errs[-1]:6.2f}%")

**Step 8: error curves (the analogue of the paper's Fig. 8).** A curve that dips and then rises means over-diffusion, so stopping early matters. A flat curve means stable behavior.

In [ ]:
plt.figure(figsize=(7, 4))
for a, (_, errs) in results.items():
    plt.plot(np.arange(len(errs)) / N_STEPS, errs, marker=".", label=f"alpha={a}")
plt.xlabel("Normalized time"); plt.ylabel("L1 error (%)"); plt.legend(); plt.grid(alpha=.3)
plt.show()

**Step 9: look at the images.** For each $\alpha$ we show the binarized result at its best stopping step and at the final step, so you can see the quality difference.

In [ ]:
fig, axes = plt.subplots(2, len(alphas), figsize=(3*len(alphas), 6))
for j, a in enumerate(alphas):
    traj, errs = results[a]
    best = errs.argmin()
    axes[0, j].imshow(traj[best] > 0.5, cmap="gray"); axes[0, j].set_title(f"a={a}, best (step {best})")
    axes[1, j].imshow(traj[-1] > 0.5, cmap="gray");   axes[1, j].set_title(f"a={a}, final")
    axes[0, j].axis("off"); axes[1, j].axis("off")
plt.tight_layout(); plt.show()

## Things to tune
- Swap in the paper's $f(u)$ (Step 3) and its parameters.
- Change `LAM` if the solution blows up (NaNs), or if it is too slow.
- Use `window=50` in `solve` for larger images.
- Pick the stopping step per $\alpha$ from Step 7 (in practice you would choose it without ground truth, e.g. a fixed step count or a convergence check).